# First test ride: is the data good enough?

This notebook checks the first ride recorded with the Bike Sensor Logger app (Pixel 7a, phone in a trouser pocket, about 28 minutes after privacy trimming). The bike had a rattling rear mudguard.

Three known events serve as a test of whether the data can show what the project needs:

1. **Road sweeper** next to the rider after a couple of minutes. It should stand out in the **sound** data.
2. **Concrete slabs** about halfway through. The joints should appear as **repeated, roughly equidistant bumps** in the accelerometer.
3. **Two short gravel sections** (about 0.5–2 km) shortly after the slabs. They should show up as **rougher vibration**.

All plots are interactive (Plotly). Drag to pan, scroll or box-select to zoom, and use the range slider under each stacked figure to move through the ride. Every overview is drawn twice, once against **time** and once against **distance**.

**How to run:** put the exported ride ZIP next to this notebook (or change `RIDE_ZIP`), then
`pip install pandas numpy scipy plotly nbformat` and run all cells.

In [ ]:
from pathlib import Path
import io, json, zipfile

from IPython.display import HTML, display

import numpy as np
import pandas as pd
from scipy import signal
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

pio.renderers.default = "notebook"   # embeds plotly.js, so the saved notebook works offline
pd.set_option("display.precision", 2)

RIDE_ZIP = Path("rides_20261019-130306.zip")
MAP_STYLE = "open-street-map"        # base map of the floating map (needs internet); "white-bg" works offline

# Known events as (from, to) in minutes since start; they are shaded in the overview plots.
# Example: EVENTS = {"concrete slabs": (12.0, 13.2), "gravel 1": (16.0, 18.5)}
EVENTS = {}

## 1. Load the ride and check data quality

Each CSV starts with `t_utc_ms`, so all three sensors share one clock. Below, `t` is seconds since the start of the trimmed ride.

In [ ]:
with zipfile.ZipFile(RIDE_ZIP) as z:
    ride = sorted({n.split("/")[0] for n in z.namelist()})[0]
    read = lambda name: pd.read_csv(io.BytesIO(z.read(f"{ride}/{name}")))
    acc, gps, snd = read("accel.csv"), read("gps.csv"), read("sound.csv")
    meta = json.loads(z.read(f"{ride}/meta.json"))

t0 = gps.t_utc_ms.iloc[0]
for df in (acc, gps, snd):
    df["t"] = (df.t_utc_ms - t0) / 1000

BAND_COLS = [c for c in snd.columns if c.startswith("band_")]
BAND_HZ = [int(c.split("_")[1]) for c in BAND_COLS]

def quality(df, name, expected_ms):
    dt = np.diff(df.t_utc_ms.values)
    return dict(sensor=name, rows=len(df), duration_min=(df.t.iloc[-1] - df.t.iloc[0]) / 60,
                rate_hz=1000 / np.median(dt), expected_hz=1000 / expected_ms,
                gaps_over_3x=int((dt > 3 * expected_ms).sum()), longest_gap_ms=dt.max())

print({k: meta[k] for k in ["ride_id", "device", "accel_sensor", "audio_source", "audio_sample_rate"]})
pd.DataFrame([quality(acc, "accelerometer", 10), quality(gps, "GPS", 1000), quality(snd, "sound", 125)])

**Reading:** no GPS or sound gaps. The accelerometer delivers about 118 Hz instead of the requested 100 Hz, which is normal on Android and slightly better. That makes the highest vibration frequency we can see about 59 Hz (the Nyquist limit), which is why the rattling mudguard (a higher-frequency buzz) isn't expected in the accelerometer.

## 2. Derived signals

* **Distance** comes from integrating the GPS (Doppler) speed. That is smoother than summing jittery positions, which overstates the length by about 5% here.
* **Elevation** is GPS altitude smoothed over 15 s. GPS altitude is noisy (±5–10 m); the phone's barometer would be much better (see recommendations).
* **Acceleration magnitude** |a| = √(ax²+ay²+az²) doesn't depend on how the phone sits in the pocket, which changes constantly. All vibration analyses use it.
* **Vibration bands** (RMS per second):
  * 0.5–3 Hz: pedalling (leg motion)
  * 3–8 Hz: body and frame motion, long waves in the surface
  * 10–50 Hz: short surface texture, the main roughness signal
* **Cadence** is the dominant frequency in the 0.6–2.2 Hz band over 8 s windows.

In [ ]:
FS = 1000 / np.median(np.diff(acc.t_utc_ms))   # actual accelerometer rate (~118 Hz)
v = gps.speed_mps.values
gps["dist_m"] = np.r_[0, np.cumsum((v[1:] + v[:-1]) / 2 * np.diff(gps.t.values))]
gps["kmh"] = gps.speed_mps * 3.6
gps["elev_m"] = gps.alt_m.rolling(15, center=True, min_periods=1).median()

to_dist = lambda t: np.interp(t, gps.t, gps.dist_m)
to_kmh = lambda t: np.interp(t, gps.t, gps.kmh)
for df in (acc, snd):
    df["dist_m"] = to_dist(df.t)

mag = np.sqrt(acc.ax**2 + acc.ay**2 + acc.az**2).values
bandpass = lambda lo, hi: signal.sosfiltfilt(signal.butter(4, [lo, hi], "bandpass", fs=FS, output="sos"), mag)
VIB_BANDS = {"pedal 0.5–3 Hz": (0.5, 3), "body 3–8 Hz": (3, 8), "texture 10–50 Hz": (10, 50)}
vib = pd.DataFrame({name: bandpass(*b) ** 2 for name, b in VIB_BANDS.items()})
vib["sec"] = acc.t.astype(int)
per_s = vib.groupby("sec").mean() ** 0.5          # RMS per second, m/s²
per_s["t"] = per_s.index + 0.5
per_s["kmh"] = to_kmh(per_s.t)
per_s["dist_m"] = to_dist(per_s.t)

# Cadence: strongest frequency between 0.6 and 2.2 Hz in 8 s windows
f_c, t_c, Z_c = signal.stft(bandpass(0.6, 2.2), FS, nperseg=int(FS * 8), noverlap=int(FS * 6))
cadence = pd.DataFrame({"t": t_c, "hz": f_c[np.argmax(np.abs(Z_c), axis=0)]})
cadence["mps"] = np.interp(cadence.t, gps.t, gps.speed_mps)
cadence["dist_m"] = to_dist(cadence.t)

# Vibration spectrogram of |a| (2.2 s windows, ~1.1 s steps)
f_a, t_a, P_a = signal.spectrogram(mag - mag.mean(), FS, nperseg=256, noverlap=128)
P_a_db = (10 * np.log10(P_a + 1e-9)).astype("float32")

# A-weighted level smoothed to 1 s for the line plots
snd["laeq_1s"] = 10 * np.log10((10 ** (snd.laeq_dbfs / 10)).rolling(8, center=True, min_periods=1).mean())
print(f"{gps.dist_m.iloc[-1]/1000:.2f} km, {gps.t.iloc[-1]/60:.1f} min, elevation {gps.elev_m.min():.0f}–{gps.elev_m.max():.0f} m, accel rate {FS:.1f} Hz")

### Linked map and cursor

The next cell adds a floating map in the bottom-right corner. Hovering over any plot below that has time or distance on its x-axis draws a vertical line at the same moment in **all** those plots and moves the marker on the map. Hovering over the map works the other way round. The map can be hidden with its button.

In [ ]:
track = dict(t=(gps.t / 60).round(4).tolist(), d=(gps.dist_m / 1000).round(4).tolist(),
             lat=gps.lat.round(6).tolist(), lon=gps.lon.round(6).tolist())
display(HTML(f"<script>{Path('ride_sync.js').read_text()}\n"
             f"initRideSync({json.dumps(dict(track=track, mapStyle=MAP_STYLE))});</script>"))

## 3. Stacked overview against time

From top to bottom:

1. **Sound heatmap:** 1/3-octave band levels every 125 ms (dBFS, uncalibrated).
2. **Sound level:** LAeq smoothed to 1 s.
3. **Vibration heatmap:** spectrogram of |a| from 0 to 59 Hz. This is the accelerometer counterpart of the sound heatmap.
4. **Vibration bands:** RMS of the three bands defined above.
5. **Speed.**
6. **Elevation.**

**Suggestion for the accelerometer:** the spectrogram (panel 3) shows *where in frequency* things happen. The band lines (panel 4) are the compact version you'd later aggregate per road segment. Section 6 adds a third view, a spatial spectrogram, which is the right tool for finding equidistant bumps.

The view opens on the first 4 minutes. Drag the range slider at the bottom to scroll.

In [ ]:
def stacked(x_snd, x_acc_spec, x_sec, x_gps, xlabel, window, kind, to_x=lambda minutes: minutes):
    fig = make_subplots(rows=6, cols=1, shared_xaxes=True, vertical_spacing=0.012,
                        row_heights=[0.24, 0.1, 0.24, 0.14, 0.1, 0.1])
    fig.add_trace(go.Heatmap(x=x_snd, y=[str(h) for h in BAND_HZ], z=snd[BAND_COLS].T.values.astype("float32"),
                             zmin=-75, zmax=-20, colorscale="Magma", colorbar=dict(title="dBFS", len=0.24, y=0.88),
                             hovertemplate="%{x:.2f}<br>%{y} Hz: %{z:.1f} dBFS<extra></extra>"), 1, 1)
    fig.add_trace(go.Scattergl(x=x_snd, y=snd.laeq_1s, name="LAeq (1 s)", line=dict(width=1)), 2, 1)
    keep = f_a <= FS / 2
    fig.add_trace(go.Heatmap(x=x_acc_spec, y=f_a[keep], z=P_a_db[keep], zmin=-50, zmax=0, colorscale="Viridis",
                             colorbar=dict(title="dB", len=0.24, y=0.47),
                             hovertemplate="%{x:.2f}<br>%{y:.1f} Hz: %{z:.1f} dB<extra></extra>"), 3, 1)
    for name in VIB_BANDS:
        fig.add_trace(go.Scattergl(x=x_sec, y=per_s[name], name=name, line=dict(width=1)), 4, 1)
    fig.add_trace(go.Scattergl(x=x_gps, y=gps.kmh, name="speed", line=dict(width=1)), 5, 1)
    fig.add_trace(go.Scattergl(x=x_gps, y=gps.elev_m, name="elevation", line=dict(width=1)), 6, 1)
    for row, title in enumerate(["band (Hz)", "LAeq dBFS", "vibration (Hz)", "RMS m/s²", "km/h", "m"], start=1):
        fig.update_yaxes(title_text=title, row=row, col=1)
    fig.update_yaxes(type="category", row=1, col=1)
    fig.update_xaxes(range=window, row=6, col=1, title_text=xlabel,
                     rangeslider=dict(visible=True, thickness=0.04))
    for label, (a, b) in EVENTS.items():
        fig.add_vrect(x0=to_x(a), x1=to_x(b), fillcolor="white", opacity=0.15, line_width=0,
                      annotation_text=label, annotation_position="top left", row="all", col=1)
    fig.update_layout(height=1100, margin=dict(l=70, r=20, t=30, b=40), dragmode="pan",
                      meta=dict(sync=[dict(xref="x", yref="paper", kind=kind)]),
                      legend=dict(orientation="h", y=1.03, x=0))
    return fig

stacked(snd.t / 60, t_a / 60, per_s.t / 60, gps.t / 60, "minutes since start", [0, 4], "time").show()

## 4. The same against distance

Distance is the natural axis for a map, because a stretch of road keeps its position however fast you ride. Stops compress to a single point. The view opens on the first 1.5 km.

In [ ]:
stacked(snd.dist_m / 1000, to_dist(t_a) / 1000, per_s.dist_m / 1000, gps.dist_m / 1000,
        "km since start", [0, 1.5], "dist", to_x=lambda m: to_dist(m * 60) / 1000).show()

## 5. Event 1: the road sweeper (sound)

We look for the loudest 10 s in the first 5 minutes, then compare its spectrum with the median spectrum of the rest of the ride at similar speed. This keeps wind noise, which grows with speed, from biasing the comparison.

In [ ]:
snd["kmh"] = to_kmh(snd.t)
s10 = snd.assign(win=(snd.t // 10).astype(int))
lin = lambda db: 10 ** (db / 10)
win_level = s10.groupby("win").laeq_dbfs.apply(lambda x: 10 * np.log10(lin(x).mean()))
early = win_level[win_level.index < 30]
w = early.idxmax()
event = s10[s10.win == w]
v_ev = event.kmh.mean()
baseline = snd[(snd.kmh.between(v_ev - 5, v_ev + 5)) & ((snd.t < w * 10 - 30) | (snd.t > w * 10 + 40))]

spec_event = 10 * np.log10(lin(event[BAND_COLS]).mean())
spec_base = 10 * np.log10(lin(baseline[BAND_COLS]).median())
print(f"Loudest 10 s in the first 5 min: {w*10/60:.1f}–{(w+1)*10/60:.1f} min ({event.dist_m.mean()/1000:.2f} km), "
      f"LAeq {early.max():.1f} dBFS vs ride median {win_level.median():.1f} dBFS (+{early.max()-win_level.median():.1f} dB), "
      f"speed {v_ev:.0f} km/h")

fig = make_subplots(rows=1, cols=2, column_widths=[0.45, 0.55],
                    subplot_titles=["Spectrum: event vs same-speed baseline", "Zoom: ±1 minute around the event"])
fig.add_trace(go.Scatter(x=[str(h) for h in BAND_HZ], y=spec_base, name=f"baseline ({v_ev-5:.0f}–{v_ev+5:.0f} km/h)"), 1, 1)
fig.add_trace(go.Scatter(x=[str(h) for h in BAND_HZ], y=spec_event, name="sweeper window"), 1, 1)
zoom = snd[snd.t.between(w * 10 - 60, w * 10 + 70)]
fig.add_trace(go.Heatmap(x=zoom.t / 60, y=[str(h) for h in BAND_HZ], z=zoom[BAND_COLS].T.values, zmin=-75, zmax=-20,
                         colorscale="Magma", showscale=False), 1, 2)
fig.update_xaxes(title_text="band (Hz)", type="category", row=1, col=1)
fig.update_yaxes(title_text="dBFS", row=1, col=1)
fig.update_xaxes(title_text="minutes", row=1, col=2)
fig.update_yaxes(type="category", row=1, col=2)
fig.update_layout(height=420, legend=dict(orientation="h", y=-0.25),
                  meta=dict(sync=[dict(xref="x2", yref="y2 domain", kind="time")], mixed=True))
fig.show()
pd.DataFrame({"band_hz": BAND_HZ, "excess_dB": (spec_event - spec_base).round(1)}).set_index("band_hz").T

**Reading:** the sweeper is found at about 1.0–1.2 min (0.5 km) without being told where to look. It is the loudest moment of the ride, about 12–13 dB above the median. Compared with riding at the same speed elsewhere, the excess grows with frequency: a few dB in the wind-dominated bands below 100 Hz, but +10 to +30 dB from 250 Hz upwards, peaking above 8 kHz (brushes and suction). That spectral shape is what a classifier would use to tell machinery from wind or traffic. **Sound data is good enough for this kind of event.**

## 6. Event 2: concrete slabs (equidistant bumps)

Slab joints repeat every *L* metres, so they produce a fixed **spatial frequency** of 1/*L* cycles per metre, however fast you ride. Pedalling produces bumps that repeat in *time* (cadence *f*), so in space they move with speed: *f*/*v*, 2*f*/*v*, … cycles per metre.

The plot below is a **spatial spectrogram**:
1. Take the envelope of the >15 Hz vibration (the bump impacts).
2. Resample it onto a 10 cm distance grid.
3. Run an FFT over 40 m windows.

The lines mark the first three cadence harmonics. **Joints show up as a horizontal stripe that does not follow those lines.**

In [ ]:
hp = signal.sosfiltfilt(signal.butter(4, 15, "highpass", fs=FS, output="sos"), mag)
env = signal.sosfiltfilt(signal.butter(2, 8, "lowpass", fs=FS, output="sos"), np.abs(hp))
grid = np.arange(0, acc.dist_m.iloc[-1], 0.1)
env_d = np.interp(grid, acc.dist_m, env)

WIN, STEP, NFFT = 400, 100, 2048                  # 40 m windows, 10 m steps, 0.1 m grid
k = np.fft.rfftfreq(NFFT, 0.1)
cols, centers = [], []
for i in range(0, len(env_d) - WIN, STEP):
    seg = env_d[i:i + WIN]
    F = np.abs(np.fft.rfft((seg - seg.mean()) * np.hanning(WIN), n=NFFT)) ** 2
    cols.append(F / (F[1:].sum() + 1e-12))
    centers.append(grid[i + WIN // 2])
spatial = 10 * np.log10(np.array(cols).T + 1e-9)
centers = np.array(centers)
ksel = (k > 0.1) & (k < 2.5)

moving = cadence[cadence.mps > 2]
fig = go.Figure(go.Heatmap(x=centers / 1000, y=k[ksel], z=spatial[ksel].astype("float32"), zmin=-45, zmax=-15,
                           colorscale="Magma", colorbar=dict(title="dB"),
                           hovertemplate="%{x:.2f} km<br>%{y:.2f} cycles/m<extra></extra>"))
for n in (1, 2, 3):
    fig.add_trace(go.Scattergl(x=moving.dist_m / 1000, y=n * moving.hz / moving.mps, mode="markers",
                               marker=dict(size=2, color="cyan"), name=f"{n}× cadence"))
fig.update_layout(height=450, xaxis_title="km", yaxis_title="cycles per metre", yaxis_range=[0.1, 2.5],
                  meta=dict(sync=[dict(xref="x", yref="paper", kind="dist")]),
                  dragmode="pan", legend=dict(orientation="h", y=1.08))
fig.show()

Next, the same check as a table. For every 250 m we find the strongest spatial period between 0.7 and 4 m and test whether it matches a cadence harmonic (within 8%). Rows that **don't** match are candidates for surface periodicity.

In [ ]:
rows = []
for start in np.arange(0, grid[-1] - 250, 250):
    m = (grid >= start) & (grid < start + 250)
    seg = env_d[m]
    F = np.abs(np.fft.rfft((seg - seg.mean()) * np.hanning(len(seg)), n=16384)) ** 2
    kk = np.fft.rfftfreq(16384, 0.1)
    sel = (kk > 0.25) & (kk < 1.4)
    i = np.argmax(F[sel])
    c = moving[(moving.dist_m >= start) & (moving.dist_m < start + 250)]
    if len(c) == 0:
        continue
    kc = (c.hz / c.mps).median()
    kp = kk[sel][i]
    harmonic = min(range(1, 5), key=lambda n: abs(kp / (n * kc) - 1))
    rows.append(dict(km=start / 1000, minute=np.interp(start + 125, gps.dist_m, gps.t) / 60,
                     kmh=np.interp(start + 125, gps.dist_m, gps.kmh), period_m=1 / kp,
                     peak_prominence=F[sel][i] / np.median(F[sel]), cadence_hz=c.hz.median(),
                     nearest_harmonic=harmonic, mismatch=abs(kp / (harmonic * kc) - 1)))
period = pd.DataFrame(rows)
period["not_pedalling"] = period.mismatch > 0.08
period.style.format(precision=2).apply(
    lambda r: ["background-color: #fde68a" if r.not_pedalling else "" for _ in r], axis=1)

**Reading:** most of the ride, the strongest repetition in the impacts is a cadence harmonic. The phone sits on the thigh, so every pedal stroke produces a bump, and on steady stretches (constant speed and cadence) those even look like a horizontal stripe.

The yellow rows are where the spatial period does not match pedalling:
* the first 1.25 km, where speed changes a lot and the 8 s cadence estimate lags behind;
* 5.75–6.0 km (minutes 14.5–15.4), slow and hardly pedalling, with a weak 1.6 m period;
* 7.0–7.5 km (minutes 18.7–21), a slow climb with a period of about 1.5–2.4 m.

None of them shows the strong, sustained stripe that slab joints should produce. Whether one of them is the slab stretch needs your memory of the route: check the map and the minute column, then enter it in `EVENTS` at the top.

**Verdict:** equidistant joints are *not reliably* separable from pedalling with the phone in a trouser pocket. The method works (it is speed-independent by construction), but the pocket adds a strong, periodic interference at exactly the spatial scales of slabs (2–6 m).

## 7. Event 3: gravel (rougher vibration)

Raw vibration rises with speed, so we normalise it. A log-log fit of band RMS against speed over the whole ride gives the expected level at each speed. The residual (in dB) says how much rougher a stretch is than an average stretch at the same speed. We do this per 100 m for the 3–8 Hz and 10–50 Hz bands and look for sustained stretches (≥300 m) that are clearly above zero.

In [ ]:
moving_s = per_s[per_s.kmh > 6].copy()
for name in ["body 3–8 Hz", "texture 10–50 Hz"]:
    fit = np.polyfit(np.log(moving_s.kmh), np.log(moving_s[name]), 1)
    moving_s[name + " excess dB"] = 20 * (np.log10(moving_s[name]) - np.polyval(fit, np.log(moving_s.kmh)) / np.log(10))
    print(f"{name}: RMS ∝ speed^{fit[0]:.2f}")

moving_s["seg"] = (moving_s.dist_m // 100).astype(int)
seg = moving_s.groupby("seg").agg(km=("dist_m", lambda d: d.min() / 1000), minute=("t", lambda t: t.mean() / 60),
                                  kmh=("kmh", "mean"),
                                  body_dB=("body 3–8 Hz excess dB", "median"),
                                  texture_dB=("texture 10–50 Hz excess dB", "median"))
seg["rough_dB"] = seg[["body_dB", "texture_dB"]].mean(axis=1)
smooth = seg.rough_dB.rolling(3, center=True, min_periods=1).mean()

fig = make_subplots(rows=3, cols=1, shared_xaxes=True, vertical_spacing=0.04, row_heights=[0.5, 0.25, 0.25])
fig.add_trace(go.Bar(x=seg.km, y=seg.body_dB, name="3–8 Hz excess", marker_color="#9ca3af", opacity=0.6), 1, 1)
fig.add_trace(go.Bar(x=seg.km, y=seg.texture_dB, name="10–50 Hz excess", marker_color="#60a5fa", opacity=0.6), 1, 1)
fig.add_trace(go.Scatter(x=seg.km, y=smooth, name="combined, 300 m mean", line=dict(color="black")), 1, 1)
fig.add_trace(go.Scatter(x=gps.dist_m / 1000, y=gps.kmh, name="speed", line=dict(width=1)), 2, 1)
fig.add_trace(go.Scatter(x=gps.dist_m / 1000, y=gps.elev_m, name="elevation", line=dict(width=1)), 3, 1)
fig.update_layout(barmode="overlay", height=650, legend=dict(orientation="h", y=1.06),
                  meta=dict(sync=[dict(xref="x", yref="paper", kind="dist")]))
fig.update_yaxes(title_text="dB vs same speed", row=1, col=1)
fig.update_yaxes(title_text="km/h", row=2, col=1)
fig.update_yaxes(title_text="m", row=3, col=1)
fig.update_xaxes(title_text="km", row=3, col=1)
fig.show()

# Sustained rough stretches: combined excess > 1.5 dB over at least 3 consecutive 100 m segments
rough = smooth > 1.5
runs = (rough != rough.shift()).cumsum()
cand = seg[rough].groupby(runs[rough]).agg(from_km=("km", "min"), to_km=("km", lambda k: k.max() + 0.1),
                                           from_min=("minute", "min"), to_min=("minute", "max"),
                                           kmh=("kmh", "mean"), rough_dB=("rough_dB", "mean"))
cand = cand[(cand.to_km - cand.from_km) >= 0.3]
cand

**Reading:**
* The table lists the sustained rough stretches the rule finds: about 2.4–2.8 km (minutes 5.7–6.5) and 8.2–8.5 km (minutes 24.5–25.3), both 2–5 dB above average for their speed.
* The slow, partly uphill section from about 6.3 to 7.7 km (minutes 16–21.5) is mildly rougher (+1 to +2 dB, mostly in 3–8 Hz). It is also clearly quieter in the sound data, which fits paths away from traffic.
* The 3–8 Hz band hardly grows with speed (exponent ≈ 0.05) while 10–50 Hz does (≈ 0.4), so the two bands carry different information.
* The per-100 m values scatter by ±3 dB even on uniform roads, so differences of 1–2 dB are not yet meaningful.

**Verdict:** **inconclusive.** Some stretches stand out after speed normalisation, but without knowing where the gravel actually was, we can't tell whether these are gravel, bad asphalt, kerbs or changes in how the phone sat in the pocket. With one ride, speed, the climb and the surface also change together.

## 8. What the pocket does to the signals

Two side effects matter for the project design:

In [ ]:
# 1) Share of vibration energy per band over the whole ride
x = mag - mag.mean()
f, P = signal.welch(x, FS, nperseg=1024)
share = {name: P[(f >= lo) & (f < hi)].sum() / P[(f >= 0.5)].sum() for name, (lo, hi) in
         {"0.5–3 Hz (pedalling)": (0.5, 3), "3–8 Hz": (3, 8), "8–20 Hz": (8, 20), "20–59 Hz": (20, FS / 2)}.items()}
print("Share of vibration energy:", {k: f"{v:.0%}" for k, v in share.items()})

# 2) Does the sound follow the pedal strokes? Compare the 1–4 kHz level envelope's spectrum with the cadence.
hf = 10 * np.log10(lin(snd[[c for c, h in zip(BAND_COLS, BAND_HZ) if 1000 <= h <= 4000]]).sum(axis=1))
rows = []
for T in np.arange(0, snd.t.iloc[-1] - 20, 20):
    m = snd.t.between(T, T + 20)
    seg_ = hf[m] - hf[m].mean()
    F = np.abs(np.fft.rfft(seg_ * np.hanning(m.sum()), n=1024)) ** 2
    ff = np.fft.rfftfreq(1024, 0.125)
    sel = (ff > 0.6) & (ff < 3.5)
    cad = cadence[cadence.t.between(T, T + 20)].hz.median()
    peak = ff[sel][np.argmax(F[sel])]
    rows.append(abs(peak / cad - round(peak / cad)) < 0.08 if cad > 0 else False)
print(f"Sound 1–4 kHz envelope peaks at a cadence harmonic in {np.mean(rows):.0%} of 20 s windows")

# 3) Coasting is quieter
coast = per_s["pedal 0.5–3 Hz"] < 0.6
s_sec = snd.groupby(snd.t.astype(int)).laeq_dbfs.mean()
print(f"LAeq while pedalling {s_sec[~coast.reindex(s_sec.index, fill_value=False)].median():.1f} dBFS, "
      f"while not pedalling {s_sec[coast.reindex(s_sec.index, fill_value=False)].median():.1f} dBFS")

* **Pedalling dominates the vibration.** About two thirds of the energy sits at 0.5–3 Hz, and its harmonics reach well into the bands we use for roughness.
* **The microphone hears the pedal strokes.** In most windows the 1–4 kHz level pulses at the cadence (fabric rubbing on the phone, plus the mudguard). Without pedalling the ride is much quieter. So part of what we would call "background noise" is the rider, not the environment.

## 9. Conclusions

| Question | Verdict | Why |
|---|---|---|
| Recording reliable? | **Yes** | No gaps, sensible rates, 28 min in a pocket with the screen off |
| Background noise (sweeper) | **Yes** | +10 dB, clearly visible across the spectrum |
| Gravel / roughness | **Inconclusive** | A few stretches are 2–5 dB rougher than usual for their speed, but the noise between 100 m segments is ±3 dB and there's no ground truth yet |
| Slab joints (periodic bumps) | **Not reliably** | Pedalling makes bumps at the same spatial scale; the method is right, the pocket is the problem |

**Recommendations before friends collect data**

1. **Mount the phone, or at least fix where it sits.** A handlebar mount or a frame/top-tube bag removes most of the pedalling signal and the fabric noise. If friends must use a pocket, record the placement for each ride so pocket rides can be modelled separately.
2. **Add an event-marker button to the app.** One tap writes a timestamped label ("gravel starts", "slabs", "sweeper"). This gives cheap ground truth without needing to remember times.
3. **Log the barometer** (much better elevation than GPS) **and the gyroscope** (helps separate leg motion from road vibration).
4. **Tell the notebook where the events were.** Fill in `EVENTS` (minutes) from memory or the map; the overviews then shade them and the slab and gravel checks can be read against them.
5. **Ride the same test loop again** with the mudguard fixed, once in the pocket and once mounted, and mark the events. That single comparison will tell us how much the mount buys.
6. **Ground truth from OpenStreetMap:** match the track to OSM `surface=*` / `smoothness=*` tags to label gravel, asphalt and concrete automatically.